# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset from the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset loaded: {metadata.name}\n{metadata.description}")

## 2. Data Overview
Review available RecordSets and their Fields, referencing all entities by their `@id`.


In [ ]:
# List all record sets available in the dataset
record_sets = list(dataset.metadata.record_sets)
print("Available record sets (by @id):\n")
for recset in record_sets:
    print(f"@id: {recset.id}, name: {recset.name}")

# Show all fields in each record set
for recset in record_sets:
    print(f"\nRecordSet '@id': {recset.id}, name: {recset.name}")
    print("Fields (by @id):")
    for field in recset.fields:
        print(f"  - @id: {field.id}, name: {field.name}, dataType: {field.data_type}, column: {field.column.id if getattr(field, 'column', None) else 'n/a'}")

## 3. Data Extraction
Load data from specific RecordSets into DataFrames for analysis. All entities are referenced by their `@id`.


In [ ]:
dataframes = {}

# Store record set @ids for easy reference
record_set_ids = [recset.id for recset in record_sets]
print(f"Extracting record sets: {record_set_ids}")
for recset in record_sets:
    try:
        records = list(dataset.records(record_set=recset.id))
        df = pd.DataFrame(records)
        dataframes[recset.id] = df
        print(f"RecordSet @id: {recset.id} | DataFrame shape: {df.shape}")
    except Exception as e:
        print(f"Could not load records for RecordSet {recset.id}: {e}")

# Preview the first DataFrame (use the first RecordSet's @id)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns for main record set '@id': {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We will select example numeric and grouping fields by their `@id` from the earlier listing for EDA. Please adjust `numeric_field_id` and `group_field_id` below as appropriate for the dataset. This example demonstrates filtering, normalization, and grouping.

In [ ]:
# Example: Pick the first RecordSet and select numeric and group fields by their @id
record_set_id = main_record_set_id  # e.g. '@id' of the main record set

# Manually specify a numeric field and group field -- you may need to adjust based on output above
# You can use the field ids from the record set overview in section 2
# For demonstration purposes, let's attempt to select columns
df = dataframes[record_set_id]
# For example, pick two columns as numeric and group
potential_numeric = [col for col in df.columns if df[col].dtype in [float, int]]
if not potential_numeric:
    # Try to convert possible numeric columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            pass
    potential_numeric = [col for col in df.columns if df[col].dtype in [float, int]]

# Select the first numeric column
if potential_numeric:
    numeric_field_id = potential_numeric[0]
else:
    numeric_field_id = None

# Pick another for grouping (string field)
group_field_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
group_field_id = group_field_candidates[0] if group_field_candidates else None

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

if numeric_field_id:
    threshold = df[numeric_field_id].mean()  # Use mean as an arbitrary threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Grouping
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, and, if grouping is available, the aggregated means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

if group_field_id and numeric_field_id:
    plt.figure(figsize=(10, 5))
    order = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).index
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id, order=order)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=30, ha='right')
    plt.show()

## 6. Conclusion
We have demonstrated how to:
- Load and inspect metadata and record sets from a dataset with a Croissant schema
- Reference all entities (record sets, fields, columns) by their `@id`
- Extract data into pandas DataFrames for analysis
- Apply basic EDA, including filtering, normalization, and grouping
- Visualize data distributions

For further scientific analysis, refer to the complete Croissant schema and field documentation. You can adjust field selection (`@id`) or explore other record sets using this notebook template.